In [5]:
import pandas as pd
import json
from pathlib import Path
from typing import TypedDict

In [3]:
full = pd.read_csv("../files/stats/lits/per_case_summary.csv")

**Falta**:

- Ejecutar el análisis estadístico dentro de `strat_dataset` con los valores actualizados
- Estratificar el nuevo `per_case_summary` en los training, test y validation csv

In [14]:
class StratifiedSplit(TypedDict):
    """Type definition matching the expected structure of the stratification JSON."""
    train: list[str]
    val: list[str]

In [17]:
def load_stratified_split(json_path: str | Path) -> StratifiedSplit:
    """Load and validate a stratification split from a JSON file.

    Args:
        json_path: Path to the stratification JSON file.

    Returns:
        StratifiedSplit: A typed dictionary containing 'train', 'val', and 'test' filename lists.

    Raises:
        FileNotFoundError: If the JSON file does not exist.
        KeyError: If required split keys are missing.
        json.JSONDecodeError: If the file contains invalid JSON.
    """
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"Stratification file not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    required_keys = {"train", "val"}
    missing = required_keys - set(data.keys())
    if missing:
        raise KeyError(
            f"Stratification JSON missing required keys: {missing}. "
            f"Found keys: {list(data.keys())}"
        )

    return {
        "train": data["train"],
        "val": data["val"],
    }

In [19]:
train_test = load_stratified_split("../files/splits/LiTS_split_seed42.json")

In [24]:
train_df = full[full["case_name"].isin(train_test["train"])]
val_df = full[full["case_name"].isin(train_test["val"])]

train_df.describe()

,ct_min,ct_max,spacing_x,spacing_y,spacing_z,liver_first,liver_last,tumor_first,tumor_last,liver_voxels,...,tumour_hu_max,voxel_volume_mm3,liver_volume_ml,tumour_volume_ml,num_lesions,min_lesion_diameter_mm,max_lesion_diameter_mm,mean_lesion_diameter_mm,liver_texture_variance,liver_noise_estimate
count,79.000000,79.000000,79.000000,79.000000,79.000000,79.000000,79.000000,73.000000,73.000000,7.900000e+01,...,73.000000,79.000000,79.000000,79.000000,79.000000,73.000000,73.000000,73.000000,79.000000,79.000000
mean,-1868.810127,3505.481013,0.776258,0.776258,1.427468,219.974684,374.569620,257.068493,337.520548,2.750019e+06,...,293.520548,0.874130,1657.108173,89.056061,8.075949,9.474570,35.538454,17.540220,939.639377,24.538209
std,1598.504318,4114.010389,0.114831,0.114831,1.079781,125.636429,190.309511,142.660857,184.298453,1.517776e+06,...,491.517240,0.710714,395.273926,170.495678,12.061564,9.268701,25.041172,10.282722,524.631860,6.679704
min,-10522.000000,1023.000000,0.556641,0.556641,0.699984,6.000000,72.000000,29.000000,66.000000,4.217350e+05,...,102.000000,0.290794,1001.266079,0.000000,0.000000,0.872686,6.569261,3.223393,227.604288,11.851852
25%,-2475.000000,1568.500000,0.686524,0.686524,0.800000,105.500000,178.500000,126.000000,157.000000,1.484798e+06,...,156.000000,0.428104,1379.689291,2.783844,1.000000,4.058415,16.831436,10.461543,599.738686,20.000000
50%,-1024.000000,2173.000000,0.753906,0.753906,1.000000,260.000000,438.000000,283.000000,370.000000,2.658803e+06,...,187.000000,0.601231,1559.518031,16.426779,5.000000,6.047134,25.878020,14.695798,833.108350,23.703704
75%,-1024.000000,3071.000000,0.834961,0.834961,1.500000,329.500000,539.500000,379.000000,492.000000,3.631967e+06,...,254.000000,1.000000,1859.852213,119.119282,10.000000,11.352113,50.289234,20.631213,1189.629590,28.148148
max,-985.000000,27572.000000,1.000000,1.000000,5.000000,422.000000,650.000000,517.000000,644.000000,7.623450e+06,...,4016.000000,3.618912,2709.848000,987.656498,75.000000,41.585956,122.759193,62.692408,3465.944827,42.222222


In [27]:
print(len(train_df), len(train_test["train"]))
print(len(val_df), len(train_test["val"]))

79 79
27 27


In [29]:
train_set = set(train_test["train"])
val_set = set(train_test["val"])
test_df = full[~full["case_name"].isin(train_set | val_set)]

In [30]:
len(test_df), len(full) - len(train_df) - len(val_df)

(25, 25)

In [ ]:
train_df.to_csv('../files/stats/lits/stratified_train.csv', index=False)
val_df.to_csv('../files/stats/lits/stratified_val.csv', index=False)
test_df.to_csv('../files/stats/lits/stratified_test.csv', index=False)